In [ ]:
import os
import sys

In [ ]:
import torch
from transformers import pipeline

hf_token = ''

if torch.cuda.is_available():
    device = 0  # CUDA GPU
else:
    device = -1  # CPU

print(f"Using device: {device}")

# Load the zero-shot classification pipeline
classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=device,
    token=hf_token
)

In [ ]:
from pyutil.db_entities import *
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

db_file = 'compiled.db'
eng = create_engine(f'sqlite:///{db_file}')
Base.metadata.create_all(eng)
session = Session(eng)

In [ ]:
import pandas as pd
from sqlalchemy import select, alias

projects = session.scalars(select(Project)).all()

project_revisions = []

for i, project in enumerate(projects):
    print(f"Processing Project: {project.project_name} (ID: {project.project_id})")
    
    pr, pr2 = alias(ProposalRevision, name="pr"), alias(ProposalRevision, name="pr2")
    stmt = (select(pr)
            .outerjoin(pr2, (pr.c.proposal_id == pr2.c.proposal_id) & (pr.c.revision_index < pr2.c.revision_index))
            .where(pr.c.project_id == project.project_id)
            .where(pr2.c.proposal_id == None)  # Only get the latest revision
        )
    
    revisions = session.execute(stmt).all()
    print(f"  Found {len(revisions)} unique proposal revisions for this project.")

    project_revisions.append(revisions)

#print revs excluding content for verification
print("Sample project revisions (excluding content):")
for revs in project_revisions:
    for rev in revs[:2]:
        print(f"  Project ID: {rev.project_id}, Proposal ID: {rev.proposal_id}, Revision Index: {rev.revision_index}, Created At: {rev.created_at}, Title: {rev.title}")

df = pd.DataFrame({
    'project_id': [rev.project_id for revs in project_revisions for rev in revs],
    'project_name': [projects[rev.project_id].project_name for revs in project_revisions for rev in revs],
    'proposal_id': [rev.proposal_id for revs in project_revisions for rev in revs],
    'revision_index': [rev.revision_index for revs in project_revisions for rev in revs],
    'created_at': [rev.created_at for revs in project_revisions for rev in revs],
    'title': [rev.title for revs in project_revisions for rev in revs],
    'content': [rev.content for revs in project_revisions for rev in revs]
})

print(f"Total projects processed: {len(project_revisions)}")
print(f"Total unique proposal revisions collected: {sum(len(revs) for revs in project_revisions)}")



In [ ]:
import json

with open('./categories.json', 'r') as f:
    domains = json.load(f)

category_mapping = [category for domain in domains for category in domain['categories']]
print(category_mapping)

In [ ]:
from tqdm import tqdm

inputs = [
    f"Project: {projects[rev.project_id].project_name}\nTitle: {rev.title}\nContents:\n{rev.content}"
    for revs in project_revisions for rev in revs
]

# P2000 - 64, 4080S - 256
BATCH_SIZE = 256
threshold = 0.35
assigned_categories_list = []
assigned_scores_list = []

print(f"Processing {len(inputs)} proposals in batches of {BATCH_SIZE}...")

for i in tqdm(range(0, len(inputs), BATCH_SIZE)):
    batch_inputs = inputs[i:i + BATCH_SIZE]
    batch_results = classifier(batch_inputs, category_mapping, multi_label=True)
    
    # If batch_inputs has only 1 element, Hugging Face returns a dict instead of a list of dicts
    if isinstance(batch_results, dict):
        batch_results = [batch_results]
        
    # Parse the results for this batch
    for result in batch_results:
        assigned_labels = [
            label for label, score in zip(result['labels'], result['scores']) 
            if score >= threshold
        ]
        
        assigned_scores = [
            score for score in result['scores'] 
            if score >= threshold
        ]

        # Fallback to top choice if nothing crosses the threshold
        if not assigned_labels and result['labels']:
            assigned_labels = [result['labels'][0]]
            assigned_scores = [result['scores'][0]]
            
        assigned_categories_list.append(json.dumps(assigned_labels))
        assigned_scores_list.append(json.dumps(assigned_scores))

df['assigned_categories'] = assigned_categories_list
df['assigned_scores'] = assigned_scores_list

In [ ]:
# export to csv for later use
df_copy = df.copy()
df_copy.drop(columns=['content'], inplace=True)
df_copy.to_csv('classified_proposals.csv', index=False)

print(df.iloc[0])
df.head()